In [28]:
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
import os
import warnings
from PIL import Image

In [29]:
# Suppress runtime warnings for divisions by zero during masking
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide")
warnings.filterwarnings("ignore", message="invalid value encountered in divide")

# --- CORE ANALYSIS LOGIC ---

def get_segmented_stats(img_name, r, g, b, nir=None, is_satellite=True):
    r, g, b = r.astype(float), g.astype(float), b.astype(float)
    
    if is_satellite and nir is not None:
        nir = nir.astype(float)
        index = (nir - r) / (nir + r + 1e-5)
        threshold = 0.2
        valid_pixels = (r + g + b + nir) > 0
    else:
        index = (2.0 * g - r - b) / (2.0 * g + r + b + 1e-5)
        threshold = 0.05
        valid_pixels = (r + g + b) > 0

    plant_mask = valid_pixels & (index > threshold)
    soil_mask = valid_pixels & (index <= threshold)

    def calculate_means(mask):
        if not np.any(mask):
            return [np.nan] * (4 if is_satellite else 3)
        if is_satellite:
            return [r[mask].mean(), g[mask].mean(), b[mask].mean(), nir[mask].mean()]
        return [r[mask].mean(), g[mask].mean(), b[mask].mean()]

    def calculate_stds(mask):
        if not np.any(mask):
            return [np.nan] * (4 if is_satellite else 3)
        if is_satellite:
            return [r[mask].std(), g[mask].std(), b[mask].std(), nir[mask].std()]
        return [r[mask].std(), g[mask].std(), b[mask].std()]

    plant_colors = calculate_means(plant_mask)
    soil_colors = calculate_means(soil_mask)

    plant_stds = calculate_stds(plant_mask)
    soil_stds = calculate_stds(soil_mask)

    cols = ['Red', 'Green', 'Blue', 'NIR'] if is_satellite else ['Red', 'Green', 'Blue']
    data = {'Imagename': img_name}

    for i, col in enumerate(cols):
        data[f'Plant_{col}_Mean'] = plant_colors[i]
        data[f'Soil_{col}_Mean'] = soil_colors[i]
        data[f'Plant_{col}_Std'] = plant_stds[i]
        data[f'Soil_{col}_Std'] = soil_stds[i]

    valid_count = np.sum(valid_pixels)
    plant_count = np.sum(plant_mask)
    soil_count = np.sum(soil_mask)

    data['Valid_Pixel_Count'] = valid_count
    data['Plant_Pixel_Count'] = plant_count
    data['Soil_Pixel_Count'] = soil_count
    data['Plant_Ratio'] = plant_count / valid_count if valid_count > 0 else np.nan
    data['Soil_Ratio'] = soil_count / valid_count if valid_count > 0 else np.nan
    data['Index_Mean'] = np.nanmean(index[valid_pixels]) if np.any(valid_pixels) else np.nan
    data['Index_Std'] = np.nanstd(index[valid_pixels]) if np.any(valid_pixels) else np.nan

    return pd.Series(data)

# --- DIRECTORY WALKER ---

def run_manual_structure_analysis(base_path):
    all_results = []
    RESEARCH_SITES = {'Scottsbluff', 'Ames', 'Crawfordville', 'Lincoln'} 
    
    if not os.path.exists(base_path):
        print(f"Error: Path '{os.path.abspath(base_path)}' not found.")
        return

    for source_type in ['UAV', 'Satellite']:
        source_path = os.path.join(base_path, source_type)
        if not os.path.exists(source_path):
            continue

        print(f"Analyzing {source_type} data...")

        for root, dirs, files in os.walk(source_path):
            for file in files:
                file_path = os.path.join(root, file)
                ext = file.lower()
                
                try:
                    res = None

                    if source_type == 'Satellite' and ext.endswith(('.tif', '.tiff')):
                        res = process_satellite(file_path)

                    elif source_type == 'UAV' and ext.endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
                        res = process_uav(file_path)
                    
                    if res is not None:
                        path_parts = root.split(os.sep)
                        res['Source'] = source_type
                        res['TPS'] = path_parts[-1] if len(path_parts) > 0 else "N/A"
                        
                        detected_loc = path_parts[-2] if len(path_parts) > 1 else "Unknown"
                        res['Location'] = detected_loc if detected_loc in RESEARCH_SITES else f"Other ({detected_loc})"
                        
                        all_results.append(res)

                except Exception as e:
                    print(f"Error in {file}: {e}")

    if all_results:
        pd.DataFrame(all_results).to_csv('manual_structure_results.csv', index=False)
        print(f"Done! Results saved for {len(all_results)} images.")

# --- FILE HANDLERS ---

def process_satellite(p):
    with rasterio.open(p) as s:
        b = s.read()
        return get_segmented_stats(os.path.basename(p), b[0], b[1], b[2], b[3], True)

def process_uav(p):
    try:
        img = Image.open(p).convert("RGB")
        arr = np.array(img)

        return get_segmented_stats(
            os.path.basename(p),
            arr[:, :, 0],
            arr[:, :, 1],
            arr[:, :, 2],
            is_satellite=False
        )
    except Exception as e:
        print(f"Error in UAV file {p}: {e}")
        return None

# --- RUN ---
print("Current working directory:", os.getcwd())
print(os.listdir("../data/raw"))
if __name__ == "__main__":
    run_manual_structure_analysis("../data/raw/2022/DataPublication_final")

Current working directory: /Users/yesyesyes/Desktop/2026 corn hackathon/corn-yield-prediction-team5/notebooks
['2022', '2023', '.DS_Store']
Analyzing UAV data...
Analyzing Satellite data...
Done! Results saved for 21843 images.


In [30]:
import pandas as pd

df = pd.read_csv("manual_structure_results.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(21843, 27)
['Imagename', 'Plant_Red_Mean', 'Soil_Red_Mean', 'Plant_Red_Std', 'Soil_Red_Std', 'Plant_Green_Mean', 'Soil_Green_Mean', 'Plant_Green_Std', 'Soil_Green_Std', 'Plant_Blue_Mean', 'Soil_Blue_Mean', 'Plant_Blue_Std', 'Soil_Blue_Std', 'Valid_Pixel_Count', 'Plant_Pixel_Count', 'Soil_Pixel_Count', 'Plant_Ratio', 'Soil_Ratio', 'Index_Mean', 'Index_Std', 'Source', 'TPS', 'Location', 'Plant_NIR_Mean', 'Soil_NIR_Mean', 'Plant_NIR_Std', 'Soil_NIR_Std']


,Imagename,Plant_Red_Mean,Soil_Red_Mean,Plant_Red_Std,Soil_Red_Std,Plant_Green_Mean,Soil_Green_Mean,Plant_Green_Std,Soil_Green_Std,Plant_Blue_Mean,...,Soil_Ratio,Index_Mean,Index_Std,Source,TPS,Location,Plant_NIR_Mean,Soil_NIR_Mean,Plant_NIR_Std,Soil_NIR_Std
0,Scottsbluff-TP3-n225_22_20.PNG,128.381794,159.300391,34.151900,54.081690,155.629446,155.744613,37.102484,49.519898,116.633272,...,0.209696,0.096540,0.061227,UAV,TP3,Scottsbluff,NaN,NaN,NaN,NaN
1,Scottsbluff-TP3-n150_20_16.PNG,131.728689,163.344267,31.110955,42.560436,149.810710,167.459002,33.008186,42.412268,129.559231,...,0.521583,0.042584,0.035635,UAV,TP3,Scottsbluff,NaN,NaN,NaN,NaN
2,Scottsbluff-TP3-n75_17_2.PNG,127.019946,155.991706,40.581544,56.365570,146.730467,156.399232,42.123354,52.344149,122.762282,...,0.418038,0.050061,0.048121,UAV,TP3,Scottsbluff,NaN,NaN,NaN,NaN
3,Scottsbluff-TP3-n150_16_17.PNG,145.399191,172.101124,33.529240,48.460687,164.587375,170.607006,34.446154,45.722870,133.239729,...,0.303835,0.059592,0.048316,UAV,TP3,Scottsbluff,NaN,NaN,NaN,NaN
4,Scottsbluff-TP3-n150_18_12.PNG,125.280780,149.389820,30.716645,52.388733,144.195621,150.102511,32.564753,48.813041,122.425267,...,0.586290,0.037805,0.043442,UAV,TP3,Scottsbluff,NaN,NaN,NaN,NaN


In [31]:
parsed_df = df.copy()

def parse_image_info(name):
    import os
    stem = os.path.splitext(name)[0]
    first_part = stem.split("_")[0]
    parts = first_part.split("-")

    location = parts[0] if len(parts) > 0 else None
    tp = parts[1] if len(parts) > 1 else None
    experiment_code = parts[2] if len(parts) > 2 else None

    return pd.Series({
        "ParsedLocation": location,
        "ParsedTP": tp,
        "ExperimentCode": experiment_code
    })

parsed_cols = parsed_df["Imagename"].apply(parse_image_info)
parsed_df = pd.concat([parsed_df, parsed_cols], axis=1)

In [32]:
print(parsed_df[["ParsedLocation", "ParsedTP", "ExperimentCode"]].drop_duplicates().head(20))

     ParsedLocation ParsedTP ExperimentCode
0       Scottsbluff      TP3           n225
1       Scottsbluff      TP3           n150
2       Scottsbluff      TP3            n75
525     Scottsbluff      TP2           n225
526     Scottsbluff      TP2           n150
528     Scottsbluff      TP2            n75
1050    Scottsbluff      TP1            n75
1051    Scottsbluff      TP1           n150
1053    Scottsbluff      TP1           n225
1575        Lincoln      TP3        hybrids
2107        Lincoln      TP2        hybrids
2639        Lincoln      TP1        hybrids
3171           Ames      TP3           4233
3173           Ames      TP3           4231
3175           Ames      TP3           4232
3843           Ames      TP2           4233
3845           Ames      TP2           4232
3846           Ames      TP2           4231
4515           Ames      TP1           4232
4516           Ames      TP1           4233


In [33]:
uav_parsed = parsed_df[parsed_df["Source"] == "UAV"].copy()
sat_parsed = parsed_df[parsed_df["Source"] == "Satellite"].copy()

print(uav_parsed.shape)
print(sat_parsed.shape)

(7281, 30)
(14562, 30)


In [34]:
agg_cols = [
    "Plant_Ratio",
    "Soil_Ratio",
    "Index_Mean",
    "Index_Std",
    "Plant_Green_Mean",
    "Plant_Red_Mean",
    "Plant_Blue_Mean"
]

uav_exp_agg = (
    uav_parsed
    .groupby(["ParsedLocation", "ExperimentCode"])[agg_cols]
    .mean()
    .reset_index()
)

sat_exp_agg = (
    sat_parsed
    .groupby(["ParsedLocation", "ExperimentCode"])[agg_cols]
    .mean()
    .reset_index()
)

print(uav_exp_agg.head())
print(sat_exp_agg.head())

   ParsedLocation ExperimentCode  Plant_Ratio  Soil_Ratio  Index_Mean  \
0            Ames           4231     0.967364    0.032636    0.190993   
1            Ames           4232     0.984313    0.015687    0.193876   
2            Ames           4233     0.976146    0.023854    0.215555   
3  Crawfordsville           4351     0.974308    0.025692    0.205359   
4  Crawfordsville           4352     0.977092    0.022908    0.212347   

   Index_Std  Plant_Green_Mean  Plant_Red_Mean  Plant_Blue_Mean  
0   0.058948        100.904604       73.367196        65.216203  
1   0.059148        111.754214       82.165583        70.823537  
2   0.064868         91.458647       63.327374        55.284512  
3   0.083472        100.278609       70.310561        63.450605  
4   0.085989         97.306253       67.493409        60.404050  
   ParsedLocation ExperimentCode  Plant_Ratio  Soil_Ratio  Index_Mean  \
0            Ames           4231     0.999082    0.000918    0.738402   
1            Ames  

In [35]:
main_df = pd.read_csv("../data/raw/2022/DataPublication_final/GroundTruth/HYBRID_HIPS_V3.5_ALLPLOTS.csv")  # adjust path
print(main_df.columns)

Index(['index', 'qrCode', 'location', 'irrigationProvided',
       'nitrogenTreatment', 'poundsOfNitrogenPerAcre', 'experiment',
       'plotLength', 'block', 'row', 'range', 'plotNumber', 'genotype',
       'plantingDate', 'totalStandCount', 'daysToAnthesis', 'GDDToAnthesis',
       'yieldPerAcre'],
      dtype='object')


In [36]:
main_df = pd.read_csv("../data/raw/2022/DataPublication_final/GroundTruth/HYBRID_HIPS_V3.5_ALLPLOTS.csv")

main_df = main_df.rename(columns={
    "location": "ParsedLocation",
    "experiment": "ExperimentCode"
})

main_df["ExperimentCode"] = main_df["ExperimentCode"].astype(str)
uav_exp_agg["ExperimentCode"] = uav_exp_agg["ExperimentCode"].astype(str)
sat_exp_agg["ExperimentCode"] = sat_exp_agg["ExperimentCode"].astype(str)

print(main_df[["ParsedLocation", "ExperimentCode"]].head())

  ParsedLocation ExperimentCode
0           Ames           4231
1           Ames           4231
2           Ames           4231
3           Ames           4231
4           Ames           4231


In [38]:
merged = main_df.merge(
    uav_exp_agg,
    on=["ParsedLocation", "ExperimentCode"],
    how="left"
)

merged = merged.merge(
    sat_exp_agg,
    on=["ParsedLocation", "ExperimentCode"],
    how="left",
    suffixes=("_uav", "_sat")
)

print(merged.shape)
merged.head()

(2291, 32)


,index,qrCode,ParsedLocation,irrigationProvided,nitrogenTreatment,poundsOfNitrogenPerAcre,ExperimentCode,plotLength,block,row,...,Plant_Green_Mean_uav,Plant_Red_Mean_uav,Plant_Blue_Mean_uav,Plant_Ratio_sat,Soil_Ratio_sat,Index_Mean_sat,Index_Std_sat,Plant_Green_Mean_sat,Plant_Red_Mean_sat,Plant_Blue_Mean_sat
0,8,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,2,...,100.904604,73.367196,65.216203,0.999082,0.000918,0.738402,0.054343,650.306965,512.390083,472.284559
1,11,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,2,...,100.904604,73.367196,65.216203,0.999082,0.000918,0.738402,0.054343,650.306965,512.390083,472.284559
2,0,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,...,100.904604,73.367196,65.216203,0.999082,0.000918,0.738402,0.054343,650.306965,512.390083,472.284559
3,9,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,...,100.904604,73.367196,65.216203,0.999082,0.000918,0.738402,0.054343,650.306965,512.390083,472.284559
4,12,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,...,100.904604,73.367196,65.216203,0.999082,0.000918,0.738402,0.054343,650.306965,512.390083,472.284559


In [39]:
print(
    merged[[
        "Plant_Ratio_uav",
        "Index_Mean_uav",
        "Plant_Ratio_sat",
        "Index_Mean_sat"
    ]].isna().mean()
)

Plant_Ratio_uav    0.083806
Index_Mean_uav     0.083806
Plant_Ratio_sat    0.083806
Index_Mean_sat     0.083806
dtype: float64


In [44]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error

from xgboost import XGBRegressor

# ----------------------------
# copy merged df
# ----------------------------
model_df = merged.copy()

# ----------------------------
# target filter
# ----------------------------
model_df = model_df[model_df["yieldPerAcre"].notna()].copy()

# ----------------------------
# feature engineering
# ----------------------------
model_df["plantingDate"] = pd.to_datetime(model_df["plantingDate"], errors="coerce")
model_df["planting_month"] = model_df["plantingDate"].dt.month
model_df["planting_dayofyear"] = model_df["plantingDate"].dt.dayofyear

model_df["nitrogen_squared"] = model_df["poundsOfNitrogenPerAcre"] ** 2
model_df["nitrogen_x_irrigation"] = (
    model_df["poundsOfNitrogenPerAcre"] * model_df["irrigationProvided"].fillna(0)
)

# ----------------------------
# columns to drop
# ----------------------------
drop_cols = [
    "yieldPerAcre",
    "qrCode",
    "plotNumber",
    "totalStandCount",
    "daysToAnthesis",
    "GDDToAnthesis",
    "plantingDate",
    
]

X = model_df.drop(columns=[c for c in drop_cols if c in model_df.columns])
y = model_df["yieldPerAcre"]

# ----------------------------
# numeric / categorical columns
# ----------------------------
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# ----------------------------
# preprocessors
# ----------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# ----------------------------
# model
# ----------------------------
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# ----------------------------
# train/test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

rmse = root_mean_squared_error(y_test, preds)
print("RMSE with image features:", rmse)

Numeric columns: ['index', 'irrigationProvided', 'poundsOfNitrogenPerAcre', 'plotLength', 'block', 'row', 'range', 'Plant_Ratio_uav', 'Soil_Ratio_uav', 'Index_Mean_uav', 'Index_Std_uav', 'Plant_Green_Mean_uav', 'Plant_Red_Mean_uav', 'Plant_Blue_Mean_uav', 'Plant_Ratio_sat', 'Soil_Ratio_sat', 'Index_Mean_sat', 'Index_Std_sat', 'Plant_Green_Mean_sat', 'Plant_Red_Mean_sat', 'Plant_Blue_Mean_sat', 'planting_month', 'planting_dayofyear', 'nitrogen_squared', 'nitrogen_x_irrigation']
Categorical columns: ['ParsedLocation', 'nitrogenTreatment', 'ExperimentCode', 'genotype']
RMSE with image features: 17.87201386529921


In [45]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(20)

,feature,importance
16,num__Index_Mean_sat,0.361907
19,num__Plant_Red_Mean_sat,0.131112
10,num__Index_Std_uav,0.018114
126,cat__genotype_SYNGENTA NK0760-3111,0.017432
127,cat__genotype_WF9 X H95,0.015243
75,cat__genotype_LH195 X PHM49,0.014883
107,cat__genotype_PHP02 X LH145,0.014031
55,cat__genotype_B37 X OH43,0.013532
53,cat__genotype_B37 X H95,0.013078
123,cat__genotype_PHZ51 X LH145,0.011605


In [46]:
import joblib

joblib.dump(pipeline, "../models/xgb_pipeline_with_imagery.pkl")

['../models/xgb_pipeline_with_imagery.pkl']